# HARP-PCB v0.3 — Best-effort auto-placement with optional AI preflight

Upload an **unrouted** `.kicad_pcb` with overlapping or pre-placed parts; the tool re-places all supported footprints and auto-sizes a rectangular board. Save intermediate stages for inspection.

**New default:** placeholder/synthetic footprints and uncertain connector opening directions **do not stop a geometry-only run**. If you optionally supply an IntelCAD API key, Claude can explain issues and suggest library names/verification steps; **suggestions are advisory, never installed as footprints, and never certify the PCB for manufacturing**. Without a key or if the model fails, the placement completes with warnings.

**Safety:** Boards already containing copper routes/zones, unsupported/cutout outlines, malformed PCB files or impossible geometric constraints may still stop rather than silently corrupt data. Check datasheets, true mechanical orientation and KiCad DRC before manufacturing.


## 1 · Settings and optional IntelCAD API key

- The API key is entered through a masked prompt. Leave empty to run **without AI**, using the local safe fallback. If enabled, only flagged **reference, footprint identifier, value and pad count** are transmitted to IntelCAD; **not raw PCB, netlist, pad coordinates, or output file**.
- `STRICT_PREFLIGHT` restores the previous refusal behavior for unverified footprints/ambiguous orientations. Default `False` keeps placing while reporting warnings.
- `SHOW_STAGES` saves snapshots and allows choosing an intermediate step.


In [ ]:
# Colab configuration
SHOW_STAGES = True
SNAPSHOT_EVERY = 2000
ITERATIONS = 4000
SEED = 42
CLEARANCE_MM = 0.65
EDGE_MARGIN_MM = 2.0
STRICT_PREFLIGHT = False
ALLOW_REPLACE_NONRECTANGULAR_OUTLINE = False

# LLM is advisory ONLY: it does not modify footprints, pads, or board geometry.
USE_LLM_PREFLIGHT = True
USE_LLM_REPORT = False
MODEL = "cc/claude-haiku-5-5"  # optional: "cc/claude-opus-5-5"
BASE_URL = "https://ai-router.intelcad.ai/v1"
from getpass import getpass
AI_API_KEY = getpass("IntelCAD API key (optional, Enter to skip): ").strip() if USE_LLM_PREFLIGHT or USE_LLM_REPORT else ""
print("Preflight AI:", "enabled" if USE_LLM_PREFLIGHT and AI_API_KEY else "local fallback without API")


## 2 · Clone the public repository and install dependencies

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = Path("/content/PCB_placement")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/amirhossein-alishahi/PCB_placement.git", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[preview]"], check=True)
subprocess.run([sys.executable, "-m", "harp_pcb", "--help"], check=True, stdout=subprocess.DEVNULL)
print("Installed from:", REPO)

## 3 · Upload `.kicad_pcb` for preflight inspection

The initial report lists synthetic/placeholder footprints and any unresolved USB/SD/switch orientation. These are **warnings in best-effort mode**, not a reason to halt geometric placement. Mechanical opening directions still require a verified marker or explicit rule.


In [ ]:
from google.colab import files
import json, shutil
from pathlib import Path
uploaded = files.upload()
boards = [name for name in uploaded if name.lower().endswith('.kicad_pcb')]
if len(boards) != 1:
    raise ValueError("Upload exactly one .kicad_pcb file")
WORK = Path('/content/harp_job')
WORK.mkdir(parents=True, exist_ok=True)
SOURCE = WORK/'input.kicad_pcb'
shutil.copyfile(Path(boards[0]),SOURCE)
print('Selected:', boards[0], '| bytes:', SOURCE.stat().st_size)

## 4 · Set hard mechanical-facing rules (when needed)

This config maps **Reference → side on final board + outward-facing direction in the unrotated footprint**.

For example `"J1": {"side":"right", "outward_local":"right"}` means the connector opening at unrotated local **+X** must face the **right-hand board edge**. `"side":"any"` permits any edge but still forces the opening to face outward. With KiCad board coordinates at angle 0, right=(+X), left=(-X), top=(-Y), bottom=(+Y). Angles 90/180/270 rotate the footprint accordingly. For the local opening orientation, rely on the footprint datasheet/3D model — don't guess it from the name.

The engine may infer `outward_local` if a footprint contains a suitably positioned **`PCB Edge` marker** on `Dwgs.User`/`Cmts.User`/Fab. In other cases explicit rules are recommended. Any ambiguous access component blocks by default.

In [ ]:
EDGE_RULES = {
    # "J1": {"side": "right", "outward_local": "right", "edge_offset_mm": 0.5},
    # "J2": {"side": "bottom", "outward_local": "right", "edge_offset_mm": 0.5},
    # "SW1": {"side": "any", "outward_local": "top", "edge_offset_mm": 0.5},
}
RULE_FILE = WORK/'edge_rules.json'
RULE_FILE.write_text(json.dumps({"components": EDGE_RULES}, indent=2), encoding='utf-8')
PREFLIGHT_FILE = WORK/'preflight.json'
pre = subprocess.run([sys.executable,'-m','harp_pcb',str(SOURCE),'--direction-rules',str(RULE_FILE),
                      '--preflight-only','--report',str(PREFLIGHT_FILE)],
                     capture_output=True, text=True)
print(pre.stdout if pre.returncode == 0 else pre.stderr)
if pre.returncode:
    raise RuntimeError('Preflight failed — inspect input board and edge rules')
PREFLIGHT = json.loads(PREFLIGHT_FILE.read_text())
print('Synthetic:',len(PREFLIGHT['synthetic_footprints']),
      '| Unresolved access-direction:',PREFLIGHT['unresolved_directions'])

## 5 · Best-effort placement, optional AI footprint review

Placement runs even if synthetic footprints or unknown connector directions remain. If the AI call fails, the CLI uses local fallback automatically. **The generated PCB is a geometry-only draft**; unverified parts are still unverified and all safety warnings remain in the JSON report. The run can still stop on potentially destructive inputs (existing traces/zones, complex outlines, invalid geometry).


In [ ]:
OUTPUT = WORK/'placed.kicad_pcb'
REPORT = WORK/'placement_report.json'
PREVIEW = WORK/'placed_preview.png'
STAGES = WORK/'stages'
cmd = [sys.executable,'-m','harp_pcb',str(SOURCE),'-o',str(OUTPUT),
       '--iterations',str(ITERATIONS),'--seed',str(SEED),
       '--clearance',str(CLEARANCE_MM),'--edge-margin',str(EDGE_MARGIN_MM),
       '--direction-rules',str(RULE_FILE),
       '--report',str(REPORT),'--preview',str(PREVIEW)]
if SHOW_STAGES:
    cmd += ['--stages-dir',str(STAGES),'--snapshot-every',str(SNAPSHOT_EVERY)]
if STRICT_PREFLIGHT:
    cmd.append('--strict-preflight')
if ALLOW_REPLACE_NONRECTANGULAR_OUTLINE:
    cmd.append('--replace-nonrectangular-outline')
if USE_LLM_PREFLIGHT:
    cmd += ['--ai-preflight','--ai-model',MODEL,'--ai-base-url',BASE_URL]
# Key is transferred via this process environment, NEVER by CLI arguments or files.
env = os.environ.copy()
if AI_API_KEY:
    env['INTELCAD_API_KEY'] = AI_API_KEY
run = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(run.stdout[-6000:] if run.returncode == 0 else run.stderr[-4000:])
if run.returncode:
    raise RuntimeError('The PCB has an unsafe/unsupported structural or geometrical condition. See error above.')
RESULT = json.loads(REPORT.read_text())
REVIEW = RESULT['footprint_audit']['ai_review']
print('Output:', OUTPUT, 'size:', OUTPUT.stat().st_size, 'bytes')
print('AI status:', REVIEW['status'], '| advisory recommendations:', len(REVIEW['recommendations']))
print('Manual review required:', RESULT['review_required_components'])
for warning in RESULT['footprint_audit']['nonblocking_warnings']:
    print('WARNING:', warning)
for rec in REVIEW['recommendations']:
    print('ADVISORY:',rec['ref'],rec['diagnosis'], '| Candidate:',rec['suggested_library_footprint'],
          '| Verify:',rec['verification_steps'])


## 6 · View final and intermediate stages

In [ ]:
from IPython.display import display, Image, Markdown
if PREVIEW.exists():
    display(Image(filename=str(PREVIEW)))
if SHOW_STAGES and (STAGES/'manifest.json').exists():
    manifest = json.loads((STAGES/'manifest.json').read_text())
    print('Saved stages:', [m['name'] for m in manifest])
    import ipywidgets as widgets
    stage_choice = widgets.Dropdown(options=[(f"{i+1}: {m['name']}", i) for i,m in enumerate(manifest)],
                                    description='Stage:')
    out = widgets.Output()
    def change_stage(change=None):
        with out:
            out.clear_output(wait=True)
            chosen=manifest[stage_choice.value]
            display(Markdown(f"**{chosen['name']}** · `{chosen['pcb']}`"))
            display(Image(filename=str(STAGES/chosen['preview'])))
    stage_choice.observe(change_stage, names='value')
    display(stage_choice,out)
    change_stage()

## 7 · Optional Claude explanation of numerical metrics

This second, optional request only transmits aggregate statistics. The **preflight** AI advice above is already included in `placement_report.json`. Neither AI call modifies geometry.


In [ ]:
if USE_LLM_REPORT and AI_API_KEY:
    import urllib.request, urllib.error
    aggregate = {k: RESULT.get(k) for k in (
        'footprints','pads','nets','board_width_mm','board_height_mm','board_area_mm2',
        'min_bounding_box_gap_mm','runtime_seconds','accepted_moves')}
    aggregate['synthetic_count'] = len(RESULT['footprint_audit']['synthetic_footprints'])
    aggregate['unresolved_orientation_count'] = len(RESULT['footprint_audit']['unresolved_directions'])
    messages = [{'role':'system','content': 'Explain a PCB geometry-only placement report. Do not claim manufacturability, KiCad DRC, correct orientation, or routability. Reply in Persian.'},
                {'role':'user','content': json.dumps(aggregate)}]
    body=json.dumps({'model':MODEL,'messages':messages,'temperature':0.1,'max_tokens':700}).encode()
    req=urllib.request.Request(BASE_URL.rstrip('/')+'/chat/completions',data=body,method='POST',
              headers={'Authorization':'Bearer '+AI_API_KEY,'Content-Type':'application/json'})
    try:
        with urllib.request.urlopen(req,timeout=75) as res:
            answer=json.load(res)
        print(answer['choices'][0]['message']['content'])
    except (urllib.error.HTTPError,urllib.error.URLError,KeyError,ValueError) as exc:
        print('AI report unavailable. Placement is unaffected:',type(exc).__name__)
else:
    print('Skipped optional metrics-only LLM report (disabled or no key).')

## 8 · Download results and safety warnings

Download the geometry-only `.kicad_pcb`, stages, preflight data and AI advisories. A synthetic footprint stays synthetic even if the model suggests a replacement name. Confirm pad geometry and manufacturer datasheets, check the opening direction, run KiCad DRC and detailed routing before manufacture.


In [ ]:
from google.colab import files
import zipfile
ARCHIVE = WORK/'harp_results.zip'
with zipfile.ZipFile(ARCHIVE,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for file in [OUTPUT,REPORT,PREVIEW,PREFLIGHT_FILE,RULE_FILE]:
        if file.exists():
            z.write(file,arcname=file.name)
    if SHOW_STAGES and STAGES.exists():
        for file in STAGES.rglob('*'):
            if file.is_file():
                z.write(file,arcname='stages/'+str(file.relative_to(STAGES)))
print('Archive:', ARCHIVE,'size:',ARCHIVE.stat().st_size,'bytes')
files.download(str(ARCHIVE))